In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import linprog

class SimplexSolver:
    def __init__(self, obj, A, b, signs, mode):
        self.obj = np.array(obj, dtype=float)
        self.A_orig = A
        self.b_orig = b
        self.signs = signs
        self.mode = mode.lower()
        self.n_vars = len(obj)
        self.n_cons = len(b)
        
        # تحضير المصفوفات للسمبلكس التعليمي (تحويل كل شيء لـ <=)
        self.A_simplex = []
        self.b_simplex = []
        for i in range(self.n_cons):
            if signs[i] == '>=':
                self.A_simplex.append([-x for x in A[i]])
                self.b_simplex.append(-b[i])
            else:
                self.A_simplex.append(A[i])
                self.b_simplex.append(b[i])
        
        self.A_simplex = np.array(self.A_simplex, dtype=float)
        self.b_simplex = np.array(self.b_simplex, dtype=float)
        
        # بناء الجدول الابتدائي (Tableau)
        self.cols = [f"X{i+1}" for i in range(self.n_vars)] + [f"S{i+1}" for i in range(self.n_cons)] + ["RHS"]
        self.rows = [f"S{i+1}" for i in range(self.n_cons)] + ["Z"]
        self.table = np.zeros((self.n_cons + 1, len(self.cols)))
        
        self.table[:self.n_cons, :self.n_vars] = self.A_simplex
        self.table[:self.n_cons, self.n_vars:self.n_vars+self.n_cons] = np.eye(self.n_cons)
        self.table[:self.n_cons, -1] = self.b_simplex
        
        if self.mode == 'max':
            self.table[-1, :self.n_vars] = -self.obj
        else:
            self.table[-1, :self.n_vars] = self.obj

    def print_tableau(self, title):
        print(f"\n--- {title} ---")
        df = pd.DataFrame(self.table, columns=self.cols, index=self.rows)
        print(df.round(2).to_string())

    def solve(self):
        # 1. عرض الجداول التعليمية (للمسائل البسيطة <=)
        self.print_tableau("Initial Tableau")
        
        # 2. الحساب الدقيق باستخدام المحرك الاحترافي (لضمان صحة الناتج مع >= و =)
        c_for_scipy = self.obj if self.mode == 'min' else -self.obj
        A_ub, b_ub, A_eq, b_eq = [], [], [], []
        
        for i in range(self.n_cons):
            if self.signs[i] == '<=':
                A_ub.append(self.A_orig[i]); b_ub.append(self.b_orig[i])
            elif self.signs[i] == '>=':
                A_ub.append([-x for x in self.A_orig[i]]); b_ub.append(-self.b_orig[i])
            elif self.signs[i] == '=':
                A_eq.append(self.A_orig[i]); b_eq.append(self.b_orig[i])


        res = linprog(c_for_scipy, 
                      A_ub=A_ub if A_ub else None, b_ub=b_ub if b_ub else None,
                      A_eq=A_eq if A_eq else None, b_eq=b_eq if b_eq else None,
                      method='highs')

        if res.success:
            print("\n" + "="*30)
            print(f"✅ Final Optimal Solution ({self.mode.upper()}):")
            for i, val in enumerate(res.x):
                print(f"X{i+1} = {val:.2f}")
            print(f"Z = {res.fun if self.mode == 'min' else -res.fun:.2f}")
            print("="*30)
        else:
            print("\n❌ No feasible solution found for these constraints.")

def solve_graphical(obj, A, b, signs, mode):
    # إعداد الرسم البياني
    limit = max(b) * 1.5 if b else 20
    x = np.linspace(0, limit, 400)
    plt.figure(figsize=(8, 8))
    
    feasible_region = np.ones_like(x) * limit
    bottom_boundary = np.zeros_like(x)

    for i in range(len(A)):
        if A[i][1] != 0:
            y = (b[i] - A[i][0]*x) / A[i][1]
            y_plot = np.clip(y, 0, limit)
            plt.plot(x, y_plot, label=f'C{i+1}: {signs[i]} {b[i]}')
            
            if signs[i] == '<=':
                feasible_region = np.minimum(feasible_region, y_plot)
            elif signs[i] == '>=':
                bottom_boundary = np.maximum(bottom_boundary, y_plot)
            elif signs[i] == '=':
                plt.plot(x, y_plot, color='black', linestyle='--', linewidth=2, label=f'C{i+1} (MUST BE ON THIS LINE)')
        else:
            val = b[i]/A[i][0]
            plt.axvline(x=val, label=f'C{i+1} Vertical')

    # تظليل منطقة الحل
    plt.fill_between(x, bottom_boundary, feasible_region, 
                     where=(feasible_region >= bottom_boundary), 
                     color='gray', alpha=0.3, hatch='//', label='Feasible Area')

    plt.xlim(0, limit); plt.ylim(0, limit)
    plt.axhline(0, color='black', lw=2); plt.axvline(0, color='black', lw=2)
    plt.title(f"Graphical Solution - {mode.capitalize()} Z")
    plt.xlabel("X1"); plt.ylabel("X2")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.show()

def main():
    print("--- Professional LP Solver (Graphical & Simplex) ---")
    nv = int(input("Number of variables: "))
    nc = int(input("Number of constraints: "))
    obj = list(map(float, input("Objective Function coefficients (space separated): ").split()))
    mode = input("Maximize or Minimize? (max/min): ").lower()
    
    A, b, signs = [], [], []
    print("\nEnter constraints (Example: 1 2 <= 12):")
    for i in range(nc):
        row = input(f"Constraint {i+1}: ").split()
        A.append([float(x) for x in row[:nv]])
        signs.append(row[nv])
        b.append(float(row[nv+1]))

    print("\nChoose Method:")
    print("1. Graphical Method (Best for 2 vars)")
    print("2. Simplex Method (Tableaus & Precise Result)")
    choice = input("Enter choice (1 for Graphical Method OR 2 for Simplex Method): ")

    if choice == '1' and nv == 2:
        solve_graphical(obj, A, b, signs, mode)
        # تشغيل السمبلكس لإعطاء الناتج الرقمي الدقيق
        SimplexSolver(obj, A, b, signs, mode).solve()
    else:
        SimplexSolver(obj, A, b, signs, mode).solve()

if __name__ == "__main__":
    main()

--- DSS Solver: Final Mode ---


: 

In [ ]:
obj = [10, 20]
A = [[1, 1], [2,2]]
B = [2, 4]
sign = ["<=", "<="]
mode = "max"
method = "simplex"